In [ ]:
from google.colab import drive
import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH, "data", "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH, "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH, "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH, "reports", "tables"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH, "reports", "figures"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

model_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_10_station_feature_engineered.parquet"
    )
)

model_df["Datetime"] = pd.to_datetime(
    model_df["Datetime"]
)

model_df["StationId"] = (
    model_df["StationId"]
    .astype(str)
)

model_df = (
    model_df
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

print("Rows:", len(model_df))
print("Stations:", model_df["StationId"].nunique())
print(
    "Start:",
    model_df["Datetime"].min()
)
print(
    "End:",
    model_df["Datetime"].max()
)

In [ ]:
RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{column}_lag_{lag}h"
    for lag in [1, 2, 12]
    for column in RAW_FEATURES
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

numeric_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
)

categorical_features = [
    "StationId"
]

model_features = (
    numeric_features
    + categorical_features
)

missing_features = [
    feature
    for feature in model_features
    if feature not in model_df.columns
]

assert not missing_features, (
    f"Missing features: {missing_features}"
)

assert TARGET in model_df.columns

print("=" * 60)
print("FINAL 10-STATION FEATURE CONTRACT")
print("=" * 60)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(model_features))
print("Target:", TARGET)

In [ ]:
unique_times = np.sort(
    model_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[
    split_position - 1
]

train_df = (
    model_df[
        model_df["Datetime"] <= train_end_time
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    model_df[
        model_df["Datetime"] > train_end_time
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 60)
print("FINAL 10-STATION TRAIN / TEST SPLIT")
print("=" * 60)

print("TRAIN")
print(
    train_df["Datetime"].min(),
    "→",
    train_df["Datetime"].max()
)
print("Rows:", len(train_df))

print("\nTEST")
print(
    test_df["Datetime"].min(),
    "→",
    test_df["Datetime"].max()
)
print("Rows:", len(test_df))

print(
    "\nTemporal separation:",
    train_df["Datetime"].max()
    < test_df["Datetime"].min()
)

print(
    "Train stations:",
    train_df["StationId"].nunique()
)

print(
    "Test stations:",
    test_df["StationId"].nunique()
)

In [ ]:
X_train = train_df[
    model_features
].copy()

y_train = train_df[
    TARGET
].copy()

X_test = test_df[
    model_features
].copy()

y_test = test_df[
    TARGET
].copy()

print(
    "X_train:",
    X_train.shape
)

print(
    "X_test:",
    X_test.shape
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Classical preprocessing pipeline created.")

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model_10 = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)

linear_model_10.fit(
    X_train,
    y_train
)

linear_pred_10 = linear_model_10.predict(
    X_test
)

print("10-station Linear Regression trained.")

In [ ]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_10 = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=18,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ]
)

decision_tree_10.fit(
    X_train,
    y_train
)

tree_pred_10 = decision_tree_10.predict(
    X_test
)

print("10-station Decision Tree trained.")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_forest_10 = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=150,
                max_depth=18,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_10.fit(
    X_train,
    y_train
)

rf_pred_10 = random_forest_10.predict(
    X_test
)

print("10-station Random Forest trained.")

In [ ]:
!pip install -q xgboost

from xgboost import XGBRegressor

xgb_10 = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                eval_metric="rmse",
                random_state=42,
                n_jobs=-1,
                tree_method="hist"
            )
        )
    ]
)

xgb_10.fit(
    X_train,
    y_train
)

xgb_pred_10 = xgb_10.predict(
    X_test
)

print("10-station XGBoost trained.")

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

classical_predictions = {
    "Linear Regression": linear_pred_10,
    "Decision Tree": tree_pred_10,
    "Random Forest": rf_pred_10,
    "XGBoost": xgb_pred_10
}

classical_rows = []

for model_name, predictions in classical_predictions.items():

    classical_rows.append(
        {
            "Model": model_name,
            "MAE": mean_absolute_error(
                y_test,
                predictions
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_test,
                    predictions
                )
            ),
            "R2": r2_score(
                y_test,
                predictions
            )
        }
    )

classical_results_10 = pd.DataFrame(
    classical_rows
).sort_values(
    "RMSE"
).reset_index(drop=True)

display(
    classical_results_10
)

In [ ]:
baseline_eval = test_df[
    [
        "StationId",
        "Datetime",
        "AQI",
        TARGET
    ]
].dropna(
    subset=[
        "AQI",
        TARGET
    ]
).copy()

baseline_mae = mean_absolute_error(
    baseline_eval[TARGET],
    baseline_eval["AQI"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_eval[TARGET],
        baseline_eval["AQI"]
    )
)

baseline_r2 = r2_score(
    baseline_eval[TARGET],
    baseline_eval["AQI"]
)

print("=" * 60)
print("FINAL 10-STATION PERSISTENCE BASELINE")
print("=" * 60)

print(
    "Usable rows:",
    len(baseline_eval)
)

print(
    "MAE:",
    round(baseline_mae, 6)
)

print(
    "RMSE:",
    round(baseline_rmse, 6)
)

print(
    "R²:",
    round(baseline_r2, 6)
)

In [ ]:
import tensorflow as tf

print(
    "TensorFlow:",
    tf.__version__
)

print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

# Neural-network feature set:
# current measurements + calendar/event information.
dl_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Hour",
    "Day",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

LOOKBACK = 12

print(
    "DL feature count:",
    len(dl_features)
)

print(
    "Lookback:",
    LOOKBACK,
    "hours"
)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

dl_imputer = SimpleImputer(
    strategy="median"
)

dl_scaler = StandardScaler()

train_dl_numeric = dl_imputer.fit_transform(
    train_df[dl_features]
)

train_dl_numeric = dl_scaler.fit_transform(
    train_dl_numeric
)

all_dl_numeric = dl_imputer.transform(
    model_df[dl_features]
)

all_dl_numeric = dl_scaler.transform(
    all_dl_numeric
)

station_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_station_encoded = (
    station_encoder.fit_transform(
        train_df[["StationId"]]
    )
)

all_station_encoded = (
    station_encoder.transform(
        model_df[["StationId"]]
    )
)

all_dl_processed = np.hstack(
    [
        all_dl_numeric,
        all_station_encoded
    ]
).astype(
    "float32"
)

print(
    "DL processed matrix:",
    all_dl_processed.shape
)

In [ ]:
def build_sequence_dataset(
    df,
    processed_matrix,
    lookback,
    target_column,
    test_start
):
    df = df.copy()

    station_ids = (
        df["StationId"]
        .astype(str)
        .to_numpy()
    )

    timestamps = (
        pd.to_datetime(
            df["Datetime"]
        )
        .to_numpy()
    )

    targets = (
        df[target_column]
        .to_numpy(
            dtype="float32"
        )
    )

    X_sequences = []
    y_values = []
    metadata = []

    for station_id in np.unique(
        station_ids
    ):

        positions = np.where(
            station_ids == station_id
        )[0]

        positions = positions[
            np.argsort(
                timestamps[positions]
            )
        ]

        for i in range(
            lookback,
            len(positions)
        ):

            current_position = positions[i]

            current_time = pd.Timestamp(
                timestamps[current_position]
            )

            # Only create sequences for the final
            # future test period.
            if current_time <= test_start:
                continue

            window_positions = positions[
                i - lookback:i
            ]

            window_times = timestamps[
                window_positions
            ]

            time_diffs = np.diff(
                window_times.astype(
                    "datetime64[s]"
                ).astype(
                    np.int64
                )
            )

            if not np.all(
                time_diffs == 3600
            ):
                continue

            target_value = targets[
                current_position
            ]

            if not np.isfinite(
                target_value
            ):
                continue

            X_sequences.append(
                processed_matrix[
                    window_positions
                ]
            )

            y_values.append(
                target_value
            )

            metadata.append(
                {
                    "StationId": station_id,
                    "Datetime": current_time
                }
            )

    return (
        np.asarray(
            X_sequences,
            dtype="float32"
        ),
        np.asarray(
            y_values,
            dtype="float32"
        ),
        pd.DataFrame(metadata)
    )


X_dl_test, y_dl_test, dl_test_meta = (
    build_sequence_dataset(
        model_df,
        all_dl_processed,
        LOOKBACK,
        TARGET,
        test_df["Datetime"].min()
    )
)

print(
    "DL test sequences:",
    X_dl_test.shape
)

print(
    "DL test targets:",
    y_dl_test.shape
)

print(
    "Stations represented:",
    dl_test_meta["StationId"].nunique()
)

In [ ]:
def build_training_sequences(
    df,
    processed_matrix,
    lookback,
    target_column,
    train_end
):
    df = df.copy()

    station_ids = (
        df["StationId"]
        .astype(str)
        .to_numpy()
    )

    timestamps = (
        pd.to_datetime(
            df["Datetime"]
        )
        .to_numpy()
    )

    targets = (
        df[target_column]
        .to_numpy(
            dtype="float32"
        )
    )

    X_sequences = []
    y_values = []
    sequence_times = []

    for station_id in np.unique(
        station_ids
    ):

        positions = np.where(
            station_ids == station_id
        )[0]

        positions = positions[
            np.argsort(
                timestamps[positions]
            )
        ]

        for i in range(
            lookback,
            len(positions)
        ):

            current_position = positions[i]

            current_time = pd.Timestamp(
                timestamps[current_position]
            )

            if current_time > train_end:
                continue

            window_positions = positions[
                i - lookback:i
            ]

            window_times = timestamps[
                window_positions
            ]

            time_diffs = np.diff(
                window_times.astype(
                    "datetime64[s]"
                ).astype(
                    np.int64
                )
            )

            if not np.all(
                time_diffs == 3600
            ):
                continue

            target_value = targets[
                current_position
            ]

            if not np.isfinite(
                target_value
            ):
                continue

            X_sequences.append(
                processed_matrix[
                    window_positions
                ]
            )

            y_values.append(
                target_value
            )

            sequence_times.append(
                current_time
            )

    return (
        np.asarray(
            X_sequences,
            dtype="float32"
        ),
        np.asarray(
            y_values,
            dtype="float32"
        ),
        pd.to_datetime(
            sequence_times
        )
    )


X_dl_train_all, y_dl_train_all, dl_train_times = (
    build_training_sequences(
        model_df,
        all_dl_processed,
        LOOKBACK,
        TARGET,
        train_end_time
    )
)

print(
    "DL training sequences:",
    X_dl_train_all.shape
)

In [ ]:
sequence_order = np.argsort(
    dl_train_times.to_numpy()
)

X_dl_train_all = (
    X_dl_train_all[
        sequence_order
    ]
)

y_dl_train_all = (
    y_dl_train_all[
        sequence_order
    ]
)

dl_train_times = (
    dl_train_times[
        sequence_order
    ]
)

validation_position = int(
    len(X_dl_train_all) * 0.85
)

X_dl_train = (
    X_dl_train_all[
        :validation_position
    ]
)

y_dl_train = (
    y_dl_train_all[
        :validation_position
    ]
)

X_dl_val = (
    X_dl_train_all[
        validation_position:
    ]
)

y_dl_val = (
    y_dl_train_all[
        validation_position:
    ]
)

print(
    "DL train:",
    X_dl_train.shape
)

print(
    "DL validation:",
    X_dl_val.shape
)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

n_dl_features = (
    X_dl_train.shape[2]
)

lstm_10 = Sequential([
    LSTM(
        64,
        input_shape=(
            LOOKBACK,
            n_dl_features
        ),
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

lstm_10.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

lstm_10.summary()

In [ ]:
dl_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

lstm_history_10 = lstm_10.fit(
    X_dl_train,
    y_dl_train,
    validation_data=(
        X_dl_val,
        y_dl_val
    ),
    epochs=30,
    batch_size=256,
    callbacks=dl_callbacks,
    verbose=1
)

print("10-station LSTM training completed.")

In [ ]:
lstm_pred_10 = (
    lstm_10
    .predict(
        X_dl_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

lstm_mae_10 = mean_absolute_error(
    y_dl_test,
    lstm_pred_10
)

lstm_rmse_10 = np.sqrt(
    mean_squared_error(
        y_dl_test,
        lstm_pred_10
    )
)

lstm_r2_10 = r2_score(
    y_dl_test,
    lstm_pred_10
)

print("=" * 60)
print("10-STATION LSTM")
print("=" * 60)

print(
    "MAE:",
    round(lstm_mae_10, 6)
)

print(
    "RMSE:",
    round(lstm_rmse_10, 6)
)

print(
    "R²:",
    round(lstm_r2_10, 6)
)

In [ ]:
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Bidirectional
)

cnn_bilstm_10 = Sequential([
    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu",
        input_shape=(
            LOOKBACK,
            n_dl_features
        )
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        )
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

cnn_bilstm_10.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

cnn_bilstm_10.summary()

In [ ]:
cnn_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

cnn_bilstm_history_10 = (
    cnn_bilstm_10.fit(
        X_dl_train,
        y_dl_train,
        validation_data=(
            X_dl_val,
            y_dl_val
        ),
        epochs=30,
        batch_size=256,
        callbacks=cnn_callbacks,
        verbose=1
    )
)

print(
    "10-station CNN-BiLSTM training completed."
)

In [ ]:
cnn_pred_10 = (
    cnn_bilstm_10
    .predict(
        X_dl_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

cnn_mae_10 = mean_absolute_error(
    y_dl_test,
    cnn_pred_10
)

cnn_rmse_10 = np.sqrt(
    mean_squared_error(
        y_dl_test,
        cnn_pred_10
    )
)

cnn_r2_10 = r2_score(
    y_dl_test,
    cnn_pred_10
)

print("=" * 60)
print("10-STATION CNN-BiLSTM")
print("=" * 60)

print(
    "MAE:",
    round(cnn_mae_10, 6)
)

print(
    "RMSE:",
    round(cnn_rmse_10, 6)
)

print(
    "R²:",
    round(cnn_r2_10, 6)
)

In [ ]:
final_results_10 = pd.DataFrame({
    "Model": [
        "Persistence Baseline",
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "LSTM",
        "CNN-BiLSTM"
    ],

    "Category": [
        "Baseline",
        "Classical ML",
        "Classical ML",
        "Ensemble ML",
        "Boosting ML",
        "Deep Learning",
        "Deep Learning"
    ],

    "MAE": [
        baseline_mae,
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Linear Regression",
            "MAE"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Decision Tree",
            "MAE"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Random Forest",
            "MAE"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "XGBoost",
            "MAE"
        ].iloc[0],
        lstm_mae_10,
        cnn_mae_10
    ],

    "RMSE": [
        baseline_rmse,
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Linear Regression",
            "RMSE"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Decision Tree",
            "RMSE"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Random Forest",
            "RMSE"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "XGBoost",
            "RMSE"
        ].iloc[0],
        lstm_rmse_10,
        cnn_rmse_10
    ],

    "R2": [
        baseline_r2,
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Linear Regression",
            "R2"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Decision Tree",
            "R2"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "Random Forest",
            "R2"
        ].iloc[0],
        classical_results_10.loc[
            classical_results_10["Model"]
            == "XGBoost",
            "R2"
        ].iloc[0],
        lstm_r2_10,
        cnn_r2_10
    ]
})

final_results_10 = (
    final_results_10
    .sort_values("RMSE")
    .reset_index(drop=True)
)

final_results_10["RMSE_vs_Persistence_%"] = (
    (
        baseline_rmse
        - final_results_10["RMSE"]
    )
    / baseline_rmse
) * 100

display(
    final_results_10
)

In [ ]:
best_final_row = (
    final_results_10
    .sort_values("RMSE")
    .iloc[0]
)

best_final_model = (
    best_final_row["Model"]
)

print("=" * 60)
print("CURRENT BEST — FINAL 10-STATION BENCHMARK")
print("=" * 60)

print(
    "Model:",
    best_final_model
)

print(
    "MAE:",
    round(
        best_final_row["MAE"],
        6
    )
)

print(
    "RMSE:",
    round(
        best_final_row["RMSE"],
        6
    )
)

print(
    "R²:",
    round(
        best_final_row["R2"],
        6
    )
)

In [ ]:
extreme_threshold = (
    test_df["AQI"]
    .dropna()
    .quantile(0.90)
)

print(
    "Extreme AQI threshold:",
    extreme_threshold
)

if best_final_model == "Random Forest":
    best_predictions = rf_pred_10

elif best_final_model == "Linear Regression":
    best_predictions = linear_pred_10

elif best_final_model == "Decision Tree":
    best_predictions = tree_pred_10

elif best_final_model == "XGBoost":
    best_predictions = xgb_pred_10

elif best_final_model == "LSTM":
    best_predictions = lstm_pred_10
    # Sequence-aligned evaluation below is separate.

elif best_final_model == "CNN-BiLSTM":
    best_predictions = cnn_pred_10

else:
    best_predictions = None

# Classical models align directly to test_df.
if best_final_model in [
    "Linear Regression",
    "Decision Tree",
    "Random Forest",
    "XGBoost"
]:
    extreme_mask = (
        test_df["AQI"]
        >= extreme_threshold
    )

    extreme_actual = (
        test_df.loc[
            extreme_mask,
            TARGET
        ]
    )

    extreme_prediction = (
        pd.Series(
            best_predictions,
            index=test_df.index
        )
        .loc[extreme_mask]
    )

    print(
        "Extreme-period records:",
        len(extreme_actual)
    )

    print(
        "Extreme MAE:",
        round(
            mean_absolute_error(
                extreme_actual,
                extreme_prediction
            ),
            6
        )
    )

    print(
        "Extreme RMSE:",
        round(
            np.sqrt(
                mean_squared_error(
                    extreme_actual,
                    extreme_prediction
                )
            ),
            6
        )
    )

else:
    print(
        "Deep-learning extreme-period evaluation "
        "will be reported from sequence-aligned metadata."
    )

In [ ]:
joblib.dump(
    linear_model_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_linear_regression.joblib"
    )
)

joblib.dump(
    decision_tree_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_decision_tree.joblib"
    )
)

joblib.dump(
    random_forest_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_random_forest.joblib"
    )
)

joblib.dump(
    xgb_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_xgboost.joblib"
    )
)

print("Classical models saved.")

In [ ]:
lstm_10.save(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_lstm.keras"
    )
)

cnn_bilstm_10.save(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_cnn_bilstm.keras"
    )
)

joblib.dump(
    dl_imputer,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_dl_imputer.joblib"
    )
)

joblib.dump(
    dl_scaler,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_dl_scaler.joblib"
    )
)

joblib.dump(
    station_encoder,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_encoder.joblib"
    )
)

print("Deep-learning models and preprocessing objects saved.")

In [ ]:
final_results_path = os.path.join(
    TABLE_PATH,
    "phase12_5_final_10_station_model_comparison.csv"
)

final_results_10.to_csv(
    final_results_path,
    index=False
)

classical_results_path = os.path.join(
    TABLE_PATH,
    "phase12_5_final_10_station_classical_results.csv"
)

classical_results_10.to_csv(
    classical_results_path,
    index=False
)

print(
    "Final comparison saved:"
)

print(
    final_results_path
)

In [ ]:
phase12_5_decision = {
    "phase": (
        "Phase 12.5 - Final 10-Station Model Comparison"
    ),

    "station_count": 10,

    "forecast_target": "AQI_t_plus_1",

    "best_model": str(
        best_final_model
    ),

    "best_mae": float(
        best_final_row["MAE"]
    ),

    "best_rmse": float(
        best_final_row["RMSE"]
    ),

    "best_r2": float(
        best_final_row["R2"]
    ),

    "final_model_selected": False,

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    )
}

decision_path = os.path.join(
    OUTPUT_PATH,
    "phase12_5_model_decision.json"
)

with open(
    decision_path,
    "w"
) as f:
    json.dump(
        phase12_5_decision,
        f,
        indent=4
    )

print(
    "Decision saved:",
    decision_path
)

In [ ]:
print("=" * 75)
print("PHASE 12.5 — FINAL 10-STATION MODEL COMPARISON")
print("=" * 75)

display(
    final_results_10
)

print(
    "\nCurrent best candidate:",
    best_final_model
)

print(
    "MAE:",
    round(
        best_final_row["MAE"],
        6
    )
)

print(
    "RMSE:",
    round(
        best_final_row["RMSE"],
        6
    )
)

print(
    "R²:",
    round(
        best_final_row["R2"],
        6
    )
)

print(
    "\nNext:",
    "Phase 13 - CPCB/OGD Live Data Integration"
)

print("=" * 75)

In [ ]:
session_status = {
    "completed_phase": (
        "Phase 12.5 - Final 10-Station Model Comparison"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    ),

    "station_count": 10,

    "forecast_target": "AQI_t_plus_1",

    "current_best_model": str(
        best_final_model
    ),

    "final_model_selected": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

print("Phase 12.5 status saved.")

In [ ]:
import os
import json
import joblib

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save final 10-station leaderboard
# --------------------------------------------------

final_results_path = os.path.join(
    TABLE_PATH,
    "phase12_5_final_10_station_model_comparison.csv"
)

final_results_10.to_csv(
    final_results_path,
    index=False
)

# --------------------------------------------------
# 2. Save classical model results
# --------------------------------------------------

classical_results_10.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase12_5_final_10_station_classical_results.csv"
    ),
    index=False
)

# --------------------------------------------------
# 3. Save classical models
# --------------------------------------------------

joblib.dump(
    linear_model_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_linear_regression.joblib"
    )
)

joblib.dump(
    decision_tree_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_decision_tree.joblib"
    )
)

joblib.dump(
    random_forest_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_random_forest.joblib"
    )
)

joblib.dump(
    xgb_10,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_xgboost.joblib"
    )
)

# --------------------------------------------------
# 4. Save deep-learning models
# --------------------------------------------------

lstm_10.save(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_lstm.keras"
    )
)

cnn_bilstm_10.save(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_cnn_bilstm.keras"
    )
)

# --------------------------------------------------
# 5. Save deep-learning preprocessing
# --------------------------------------------------

joblib.dump(
    dl_imputer,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_dl_imputer.joblib"
    )
)

joblib.dump(
    dl_scaler,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_dl_scaler.joblib"
    )
)

joblib.dump(
    station_encoder,
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_encoder.joblib"
    )
)

# --------------------------------------------------
# 6. Save best-model decision
# --------------------------------------------------

best_final_row = (
    final_results_10
    .sort_values("RMSE")
    .iloc[0]
)

phase12_5_summary = {
    "completed_phase": (
        "Phase 12.5 - Final 10-Station Model Comparison"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    ),

    "station_count": 10,

    "forecast_target": "AQI_t_plus_1",

    "current_best_model": str(
        best_final_row["Model"]
    ),

    "best_mae": float(
        best_final_row["MAE"]
    ),

    "best_rmse": float(
        best_final_row["RMSE"]
    ),

    "best_r2": float(
        best_final_row["R2"]
    ),

    "status": "Production candidate",
    "final_model_selected": False
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase12_5_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase12_5_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 7. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": (
        "Phase 12.5 - Final 10-Station Model Comparison"
    ),

    "next_phase": (
        "Phase 13 - CPCB/OGD Live Data Integration"
    ),

    "current_best_model": str(
        best_final_row["Model"]
    ),

    "station_count": 10,

    "forecast_target": "AQI_t_plus_1",

    "final_model_selected": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 8. Verify important files
# --------------------------------------------------

files_to_check = [
    final_results_path,
    summary_path,
    status_path,

    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_random_forest.joblib"
    ),

    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_xgboost.joblib"
    ),

    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_lstm.keras"
    ),

    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_cnn_bilstm.keras"
    )
]

print("=" * 65)
print("PHASE 12.5 CHECKPOINT SAVED")
print("=" * 65)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCurrent best candidate:", best_final_row["Model"])
print("MAE :", round(float(best_final_row["MAE"]), 6))
print("RMSE:", round(float(best_final_row["RMSE"]), 6))
print("R²  :", round(float(best_final_row["R2"]), 6))

print("\nNext: Phase 13 - CPCB/OGD Live Data Integration")